# Gu-1B-500M-tokens

**Anewo's first experimental language model.** This notebook accompanies the 500M-token checkpoint of a randomly initialized Qwen3-style model with **1,009,866,624 trainable parameters**.

We paused at the completed 500M checkpoint because our remaining Colab compute budget could not cover the planned 1B-token training goal at the observed throughput. This release is a learning resource for inspecting training, evaluating weak and strong outputs, restoring checkpoints and preparing future experiments with larger models as compute becomes available.

The checkpoint reached 4K training sequences. The 16K and 32K stages remain planned work. Its reported held-out loss is 2.237 and perplexity is 9.37 on 32,768 targets. Sample generations showed factual errors and context contradictions.

The first five cells explain the experiment. Run code cells in order for training, or run setup, configuration, architecture and checkpoint definitions followed by generation when evaluating the published weights. Review the paid runtime rate before starting workloads.


## What we are doing

The saved release represents **500,000,000 training targets**. The notebook retains a configurable continuation goal of **1,000,000,000 targets**. Set `SESSION_STOP_TOKENS=500_000_000` before a fresh run to stop and save at the release milestone. Evaluation tokens are excluded. We retain complete documents, append EOS, reserve documents for validation and stream the corpus without first collecting billions of tokens in RAM.

For faster initial training, use **4,096-token sequences through 900M tokens**, **16,384 through 980M**, then **32,768 through 1B**. These allocations are experiment defaults. The final stage supplies practice at 32k context; held-out long-distance retrieval tests are still needed to assess usable context. Packed documents can be unrelated to each other.

Increase `TARGET_TOKENS` later and resume from a full checkpoint. The learning-rate schedule depends on consumed tokens and can continue when the target is extended.

## What we are using

* Model width 1,536, 27 layers, 24 query heads, 8 key/value heads, 64 features per head and SwiGLU width 6,144. Input/output embeddings share weights.
* SmolLM's 49,152-token tokenizer and the streaming SmolLM cosmopedia-v2 corpus. Hub revisions are resolved to commit IDs and recorded in checkpoints.
* Colab A100, BF16 autocast, Flash scaled-dot-product attention, activation checkpointing and recomputed chunks of output cross-entropy to limit memory.
* Fused AdamW on CUDA with FP32 master weights. This replaces the tutorial's custom Muon optimizer to use PyTorch's fused implementation. Speed must be measured on the actual A100.
* One sequence per microbatch, accumulating **65,536 training tokens per update**. Smaller updates occur at save milestones and context-stage boundaries so every checkpoint has an exact token count.

References: [Hugging Face streaming and recovery](https://huggingface.co/docs/datasets/stream), [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).

## Checkpoint schedule

| Training tokens consumed | Files saved |
|---|---|
| Every 100M | Model weights and architecture/tokenizer metadata |
| 500M and 1B | Full training checkpoint with FP32 weights, AdamW state, RNG state, corpus cursor, unused token buffer and settings |
| A planned session stop | Extra full checkpoint, even before 500M |

Each committed checkpoint folder includes a model file and a manifest. At full-save milestones the same model file also holds the training state. A temporary folder is published only after all its files have been written. Keep full checkpoints in trusted storage.

**Drive saving is enabled by default.** Budget roughly **40GB** to retain all ten requested milestones, excluding exports and extra session checkpoints. A weights-only save is about 2GB, and a full save is about 12GB. Drive quota and file transfer speed can affect the run. A Colab runtime deletion loses files saved only under `/content`.

## Run, stop, resume and export

1. Select **Runtime → Change runtime type → A100 GPU** and run setup.
2. Keep `SAVE_TO_DRIVE=True` to save directly to your Drive. Review the checkpoint folder and token budget.
3. Set `SESSION_STOP_TOKENS` to an absolute token count if you need an earlier full checkpoint before ending a session. Alternatively set a session time limit. Both stop at a completed update.
4. Use `RESUME_FROM="auto"` to find the latest full checkpoint for this experiment. If the folder contains several independent experiments, select the full `.pt` file explicitly. A fresh run folder is created for each training invocation so earlier milestone files remain available.
5. Re-run setup and definition cells after reconnecting, then run training to restore progress. The corpus may replay part of its current shard to recover its cursor. Use the recorded library versions for exact replay. Version changes produce a warning. Resume always uses the saved source revisions.
6. The final export cell can download a weights file or ZIP the complete resumable checkpoint. The Drive copy goes under `MyDrive/Gu-1B/checkpoints/`, where automatic resume can find it. To resume from a downloaded ZIP, extract `checkpoint.pt` and `manifest.json` together, then set `RESUME_FROM` to the extracted `.pt` path. Setup and export also work in a CPU runtime.

Training runs continuously by default. Set `SESSION_HOURS=1.0` to save a full checkpoint and stop after roughly an hour, then resume on a later session. Extra session checkpoints increase storage use. The total experiment remains 1B tokens. Network failures or forced runtime termination can still lose work since the last completed full checkpoint.

To switch an earlier Gu-1B run to this 1B-token plan, select a full checkpoint at or before 900M tokens. Resume allows a different `loss_chunk` and context-stage boundaries that are still in the future. Architecture, optimizer settings and completed context stages must match. A different loss chunk size can change floating-point rounding. Changing source cells affects the next training invocation. Before interrupting an active run, confirm a full resumable checkpoint exists. The default full saves occur at 500M and 1B tokens. Weights-only 100M snapshots do not include optimizer or streaming state.

## Setup and run settings

In [ ]:
%pip -q install "transformers>=4.45,<6" "datasets>=3,<6" huggingface_hub tqdm

import os, math, time, json, random, re, warnings, uuid, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.nn.attention import sdpa_kernel, SDPBackend
from transformers import AutoTokenizer
from datasets import load_dataset
from huggingface_hub import HfApi
from tqdm.auto import tqdm

SAVE_TO_DRIVE = True
TARGET_TOKENS = 1_000_000_000
RESUME_FROM = "auto"   # Or a full checkpoint .pt path, or None for a fresh model.
SESSION_STOP_TOKENS = None  # Absolute token count, e.g. 50_000_000.
SESSION_HOURS = None  # Set 1.0 for an optional planned full save and session exit.

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Gu-1B/checkpoints')
else:
    CHECKPOINT_ROOT = Path('/content/Gu-1B/checkpoints')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(), 'VRAM GB', torch.cuda.get_device_properties(0).total_memory / 1e9)
else:
    print('CPU runtime. Saved checkpoints can be listed, downloaded, or copied here.')
print('Versions', torch.__version__, __import__('datasets').__version__, __import__('transformers').__version__)
print('Checkpoint root', CHECKPOINT_ROOT)

## Model configuration

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 49152
    d_model: int = 1536
    n_layers: int = 27
    n_heads: int = 24
    n_kv_heads: int = 8
    d_ff: int = 6144
    max_seq_len: int = 32768
    rope_theta: float = 500000.0
    rms_norm_eps: float = 1e-6
    loss_chunk: int = 1024
    activation_checkpointing: bool = True

    def __post_init__(self):
        dimensions = (self.vocab_size, self.d_model, self.n_layers, self.n_heads,
                      self.n_kv_heads, self.d_ff, self.max_seq_len, self.loss_chunk)
        if not all(type(value) is int for value in dimensions):
            raise ValueError('Model dimensions must be integers')
        if not all(math.isfinite(v) and v > 0 for v in (self.rope_theta, self.rms_norm_eps)):
            raise ValueError('rope_theta and rms_norm_eps must be finite and positive')
        if min(self.vocab_size, self.d_model, self.n_layers, self.n_heads,
               self.n_kv_heads, self.d_ff, self.max_seq_len, self.loss_chunk) <= 0:
            raise ValueError('Model dimensions must be positive')
        if self.d_model % self.n_heads or self.n_heads % self.n_kv_heads:
            raise ValueError('Invalid attention head dimensions')
        if (self.d_model // self.n_heads) % 2:
            raise ValueError('RoPE needs an even head dimension')

@dataclass
class RunConfig:
    target_tokens: int = 1_000_000_000
    update_tokens: int = 65536
    checkpoint_every: int = 100_000_000
    resume_every: int = 500_000_000
    base_lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_tokens: int = 20_000_000
    weight_decay: float = 0.1
    stage_ends: tuple = (900_000_000, 980_000_000)
    stage_lengths: tuple = (4096, 16384, 32768)

    def __post_init__(self):
        counts = (self.target_tokens, self.update_tokens, self.checkpoint_every,
                  self.resume_every, self.warmup_tokens, *self.stage_lengths, *self.stage_ends)
        if not all(type(value) is int and value > 0 for value in counts):
            raise ValueError('Token counts and stage lengths must be positive integers')
        if not (0 <= self.min_lr <= self.base_lr and self.base_lr > 0 and
                math.isfinite(self.base_lr) and math.isfinite(self.weight_decay) and self.weight_decay >= 0):
            raise ValueError('Invalid learning rate or weight decay')
        if len(self.stage_lengths) != len(self.stage_ends) + 1:
            raise ValueError('Need one more stage length than stage boundaries')
        if any(x <= 0 for x in self.stage_lengths) or tuple(sorted(set(self.stage_ends))) != tuple(self.stage_ends):
            raise ValueError('Invalid context stages')

    def seq_len(self, tokens):
        return self.stage_lengths[sum(tokens >= end for end in self.stage_ends)]

    def lr(self, tokens):
        scale = min(1.0, max(1, tokens) / self.warmup_tokens)
        decay = math.sqrt(self.warmup_tokens / max(self.warmup_tokens, tokens))
        return max(self.min_lr, self.base_lr * scale * decay)

## Gu-1B transformer and memory-efficient loss

RoPE rotates Q and K along token positions. Each attention layer uses fewer KV heads and normalizes Q/K before causal attention. Blocks and output-loss chunks are recomputed during backward to reduce retained activations. The fixed RoPE frequency base is an experiment setting for this run.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        normalized = x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps)
        return normalized.to(x.dtype) * self.weight.to(x.dtype)

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.head_dim = c.d_model // c.n_heads
        kv_dim = c.n_kv_heads * self.head_dim
        self.q = nn.Linear(c.d_model, c.d_model, bias=False)
        self.k = nn.Linear(c.d_model, kv_dim, bias=False)
        self.v = nn.Linear(c.d_model, kv_dim, bias=False)
        self.out = nn.Linear(c.d_model, c.d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim, c.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, c.rms_norm_eps)

    def forward(self, x, cos, sin):
        b, t, _ = x.shape
        q = self.q_norm(self.q(x).view(b, t, self.c.n_heads, self.head_dim))
        k = self.k_norm(self.k(x).view(b, t, self.c.n_kv_heads, self.head_dim))
        v = self.v(x).view(b, t, self.c.n_kv_heads, self.head_dim).transpose(1, 2)
        def rotate(z):
            a, d = z.chunk(2, dim=-1)
            c, s = cos.to(z.dtype), sin.to(z.dtype)
            return torch.cat((a * c - d * s, d * c + a * s), -1).transpose(1, 2)
        q, k = rotate(q), rotate(k)
        context = sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]) if x.is_cuda else nullcontext()
        with context:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)
        return self.out(out.transpose(1, 2).contiguous().view(b, t, self.c.d_model))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c.d_model, c.rms_norm_eps)
        self.ff_norm = RMSNorm(c.d_model, c.rms_norm_eps)
        self.attention = Attention(c)
        self.gate = nn.Linear(c.d_model, c.d_ff, bias=False)
        self.up = nn.Linear(c.d_model, c.d_ff, bias=False)
        self.down = nn.Linear(c.d_ff, c.d_model, bias=False)

    def forward(self, x, cos, sin):
        x = x + self.attention(self.attn_norm(x), cos, sin)
        z = self.ff_norm(x)
        return x + self.down(F.silu(self.gate(z)) * self.up(z))

class GuModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.apply(self.initialize)

    def initialize(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def hidden(self, ids):
        if ids.shape[1] > self.config.max_seq_len:
            raise ValueError('Input exceeds configured context')
        x = self.embedding(ids)
        dim = self.config.d_model // self.config.n_heads
        inv = self.config.rope_theta ** (-torch.arange(0, dim, 2, device=ids.device).float() / dim)
        angles = torch.arange(ids.shape[1], device=ids.device).float()[:, None] * inv[None, :]
        cos, sin = angles.cos()[None, :, None, :].to(x.dtype), angles.sin()[None, :, None, :].to(x.dtype)
        for block in self.blocks:
            if self.training and self.config.activation_checkpointing:
                x = checkpoint(block, x, cos, sin, use_reentrant=False)
            else:
                x = block(x, cos, sin)
        return self.norm(x)

    def forward(self, ids):
        return F.linear(self.hidden(ids), self.embedding.weight)

    def loss_sum(self, ids, targets):
        hidden = self.hidden(ids).reshape(-1, self.config.d_model)
        targets = targets.reshape(-1)
        result = hidden.new_zeros((), dtype=torch.float32)
        def chunk_loss(h, y):
            return F.cross_entropy(F.linear(h, self.embedding.weight).float(), y,
                                   reduction='sum', ignore_index=-100)
        for start in range(0, targets.numel(), self.config.loss_chunk):
            h, y = hidden[start:start+self.config.loss_chunk], targets[start:start+self.config.loss_chunk]
            if torch.is_grad_enabled():
                result = result + checkpoint(chunk_loss, h, y, use_reentrant=False)
            else:
                result = result + chunk_loss(h, y)
        return result

with torch.device('meta'):
    parameter_preview = GuModel(ModelConfig())
parameter_count = sum(p.numel() for p in parameter_preview.parameters())
assert parameter_count == 1_009_866_624
print(f'Gu-1B trainable parameters: {parameter_count:,}')
del parameter_preview

## Reproducible streaming data

Every hundredth document is held out for validation before tokenization. Data and tokenizer commits are pinned. Checkpoints retain the streaming cursor and remaining tokens, allowing recovery within the current shard. Source exhaustion raises an error instead of silently repeating data. Keep matching library versions when resuming.

In [ ]:
def source_metadata():
    api = HfApi()
    return {
        'tokenizer_id': 'HuggingFaceTB/SmolLM-135M',
        'tokenizer_revision': api.model_info('HuggingFaceTB/SmolLM-135M').sha,
        'dataset_id': 'HuggingFaceTB/smollm-corpus',
        'dataset_revision': api.dataset_info('HuggingFaceTB/smollm-corpus').sha,
        'dataset_config': 'cosmopedia-v2',
    }

class TokenStream:
    def __init__(self, metadata, validation=False):
        self.metadata = metadata
        self.validation = validation
        self.tokenizer = AutoTokenizer.from_pretrained(metadata['tokenizer_id'], revision=metadata['tokenizer_revision'])
        if max(self.tokenizer.get_vocab().values()) + 1 != 49152:
            raise ValueError('Unexpected tokenizer vocabulary')
        self.dataset = load_dataset(metadata['dataset_id'], metadata['dataset_config'],
                                    revision=metadata['dataset_revision'], split='train', streaming=True)
        self.iterator = None
        self.buffer = []
        self.documents = 0

    def batch(self, count, device):
        if type(count) is not int or count < 1:
            raise ValueError('Batch length must be a positive integer')
        if self.iterator is None:
            self.iterator = iter(self.dataset)
        while len(self.buffer) < count + 1:
            try:
                item = next(self.iterator)
            except StopIteration:
                raise RuntimeError('Corpus exhausted. Choose a larger source for continued training.')
            held_out = self.documents % 100 == 0
            self.documents += 1
            if held_out != self.validation:
                continue
            ids = self.tokenizer.encode(item['text'], add_special_tokens=False)
            self.buffer.extend(ids + [self.tokenizer.eos_token_id])
        tokens = torch.tensor(self.buffer[:count+1], dtype=torch.long, device=device)[None, :]
        del self.buffer[:count]
        return tokens[:, :-1], tokens[:, 1:]

    def state_dict(self):
        return {'source': self.metadata, 'validation': self.validation,
                'cursor': self.dataset.state_dict(), 'buffer': self.buffer.copy(),
                'documents': self.documents}

    def load_state_dict(self, state):
        if state['source'] != self.metadata or state['validation'] != self.validation:
            raise ValueError('Checkpoint source differs from the active stream')
        self.dataset.load_state_dict(state['cursor'])
        self.iterator = iter(self.dataset)
        self.buffer = list(state['buffer'])
        self.documents = state['documents']

## Checkpoint storage and recovery

At a full milestone, `checkpoint.pt` contains the FP32 model and complete training state. At other milestones it contains BF16 model weights for inference. Every file includes architecture, tokenizer/corpus revisions, token count and library versions. Only committed folders with a manifest are discovered automatically.

In [ ]:
def versions():
    return {'torch': str(torch.__version__), 'datasets': __import__('datasets').__version__,
            'transformers': __import__('transformers').__version__}

def checkpoint_files(root):
    result = []
    for manifest in Path(root).glob('run-*/tokens-*/manifest.json'):
        directory = manifest.parent
        if not re.fullmatch(r'tokens-[0-9]{12,}', directory.name):
            continue
        path = directory / 'checkpoint.pt'
        try:
            info = json.loads(manifest.read_text(encoding='utf-8'))
            if (type(info['tokens']) is not int or info['tokens'] != int(directory.name[7:]) or
                    type(info['resumable']) is not bool or type(info['bytes']) is not int or
                    info['bytes'] <= 0 or path.stat().st_size != info['bytes']):
                raise ValueError('inconsistent checkpoint metadata or file size')
            modified = path.stat().st_mtime_ns
        except (OSError, UnicodeError, TypeError, KeyError, ValueError) as error:
            warnings.warn(f'Skipping unreadable checkpoint {directory}: {error}')
            continue
        result.append((info, path, modified))
    ordered = sorted(result, key=lambda row: (row[0]['tokens'], row[2]))
    return [(info, path) for info, path, _ in ordered]

def checkpoint_payload(path, *, weights_only):
    # Page tensors in as needed, especially the FP32 weights of a large training save.
    return torch.load(path, map_location='cpu', mmap=True, weights_only=weights_only)

def latest_full(root):
    entries = checkpoint_files(root)
    if len({info.get('training_id', 'legacy') for info, _ in entries}) > 1:
        raise ValueError('This folder contains independent training runs. Set RESUME_FROM to the intended full checkpoint.pt.')
    full = [path for info, path in entries if info['resumable']]
    return full[-1] if full else None

def rng_state():
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    if state['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state['cuda'])

def save_checkpoint(folder, model, optimizer, data, run, tokens, updates, full):
    target = Path(folder) / f'tokens-{tokens:012d}'
    if target.exists():
        raise FileExistsError(f'Checkpoint already exists: {target}')
    temporary = target.with_name(target.name + '.incomplete-' + uuid.uuid4().hex[:8])
    temporary.mkdir(parents=True)
    dtype = torch.float32 if full else torch.bfloat16
    weights = {name: value.detach().to(device='cpu', dtype=dtype)
               for name, value in model.state_dict().items()}
    payload = {'name': 'Gu-1B', 'config': asdict(model.config), 'weights': weights,
               'tokens': tokens, 'updates': updates, 'resumable': full,
               'source': data.metadata, 'run': asdict(run), 'versions': versions(), 'training_id': model.training_id}
    if full:
        payload.update(optimizer=optimizer.state_dict(), data=data.state_dict(), rng=rng_state())
    path = temporary / 'checkpoint.pt'
    torch.save(payload, path)
    info = {'tokens': tokens, 'updates': updates, 'resumable': full,
            'training_id': model.training_id, 'bytes': path.stat().st_size}
    (temporary / 'manifest.json').write_text(json.dumps(info, indent=2))
    temporary.rename(target)
    print(f'Saved {tokens:,} tokens, resumable={full}: {target}', flush=True)
    return target / 'checkpoint.pt'

## Token-counted training

Save boundaries count successful training targets. Updates are shortened to land exactly on milestones. BF16 uses FP32 AdamW parameters. The learning-rate floor is 3e-5. The default 20M-token warmup followed by inverse-square-root decay continues consistently when the total budget grows. A non-finite gradient stops the run before updating. Session checkpoints are saved at completed optimizer updates.

In [ ]:
def train_gu(config, run, root, resume_from=None, session_stop=None, session_hours=None, device='cuda', data_factory=TokenStream):
    device = torch.device(device)
    if session_stop is not None and (type(session_stop) is not int or session_stop <= 0):
        raise ValueError('SESSION_STOP_TOKENS must be a positive integer or None')
    if session_hours is not None and (not math.isfinite(session_hours) or session_hours <= 0):
        raise ValueError('SESSION_HOURS must be finite and positive or None')
    if device.type == 'cuda':
        check_training_gpu(device, config)
    if max(run.stage_lengths) > config.max_seq_len:
        raise ValueError('Context stage exceeds model context')
    saved = None
    if resume_from is not None:
        # Full recovery includes Python/NumPy RNG objects. Load only your own checkpoints.
        saved = checkpoint_payload(resume_from, weights_only=False)
        if not saved.get('resumable', False):
            raise ValueError('Choose a full resumable checkpoint')
        previous_config, current_config = dict(saved['config']), asdict(config)
        previous_chunk = previous_config.pop('loss_chunk')
        current_config.pop('loss_chunk')
        if previous_config != current_config:
            raise ValueError('Model settings changed. Restore the saved ModelConfig.')
        if previous_chunk != config.loss_chunk:
            warnings.warn('Resuming with a different loss chunk size. Floating-point rounding may differ.')
        if saved['versions'] != versions():
            warnings.warn(f"Library versions differ from the saved run: {saved['versions']}. "
                          "Exact replay can depend on the original versions.")
        if run.target_tokens < saved['tokens']:
            raise ValueError('TARGET_TOKENS is below the selected checkpoint token count')
        previous_run, current_run = dict(saved['run']), asdict(run)
        previous_run.pop('target_tokens'); current_run.pop('target_tokens')
        previous_ends = previous_run.pop('stage_ends')
        current_ends = current_run.pop('stage_ends')
        if previous_run != current_run:
            raise ValueError('Only target_tokens and future stage boundaries may change when resuming')
        if any(old != new and min(old, new) < saved['tokens']
               for old, new in zip(previous_ends, current_ends)):
            raise ValueError('A changed context boundary is before saved progress. Keep the saved stage_ends.')
    metadata = saved['source'] if saved else source_metadata()
    data = data_factory(metadata)
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    with torch.device(device):
        model = GuModel(config)
    model.training_id = saved.get('training_id', 'legacy') if saved else uuid.uuid4().hex
    model.source_metadata = metadata
    model.last_checkpoint = Path(resume_from) if resume_from else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=run.base_lr,
                                  weight_decay=run.weight_decay, fused=device.type=='cuda')
    tokens, updates = 0, 0
    if saved:
        model.load_state_dict(saved['weights'])
        optimizer.load_state_dict(saved['optimizer'])
        data.load_state_dict(saved['data'])
        tokens, updates = saved['tokens'], saved['updates']
        restore_rng(saved['rng'])
        del saved
    if tokens >= run.target_tokens:
        print(f'Already at {tokens:,} tokens. Increase target_tokens to continue.')
        return model, tokens
    if session_stop is not None and session_stop <= tokens:
        raise ValueError('SESSION_STOP_TOKENS must exceed current progress')
    folder = Path(root) / ('run-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6])
    folder.mkdir(parents=True)
    model.train()
    start = time.monotonic()
    start_tokens = tokens
    stop = min(run.target_tokens, session_stop or run.target_tokens)
    progress = tqdm(total=run.target_tokens, initial=tokens, unit='tok', unit_scale=True)
    while tokens < stop:
        next_model = (tokens // run.checkpoint_every + 1) * run.checkpoint_every
        next_full = (tokens // run.resume_every + 1) * run.resume_every
        future_stages = [end for end in run.stage_ends if end > tokens]
        boundary = min([stop, next_model, next_full] + future_stages)
        update_size = min(run.update_tokens, boundary - tokens)
        optimizer.zero_grad(set_to_none=True)
        lr = run.lr(tokens + update_size)
        for group in optimizer.param_groups:
            group['lr'] = lr
        remaining, loss_value = update_size, 0.0
        while remaining:
            size = min(run.seq_len(tokens), remaining)
            x, y = data.batch(size, device)
            amp = torch.autocast('cuda', dtype=torch.bfloat16) if device.type=='cuda' else nullcontext()
            with amp:
                loss = model.loss_sum(x, y)
            (loss / update_size).backward()
            loss_value += loss.detach().item()
            remaining -= size
        norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not torch.isfinite(norm):
            raise FloatingPointError('Non-finite gradient. Resume the last full checkpoint and inspect the run.')
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        tokens += update_size
        updates += 1
        progress.update(update_size)
        progress.set_postfix(loss=f'{loss_value/update_size:.3f}', lr=f'{lr:.2e}', ctx=run.seq_len(tokens-1))
        timed_stop = session_hours is not None and time.monotonic() - start >= session_hours * 3600
        full = tokens % run.resume_every == 0 or tokens == stop or timed_stop
        if tokens % run.checkpoint_every == 0 or full:
            model.last_checkpoint = save_checkpoint(folder, model, optimizer, data, run, tokens, updates, full)
        if timed_stop:
            break
    progress.close()
    rate = (tokens-start_tokens) / max(1, time.monotonic()-start)
    print(f'Gu-1B reached {tokens:,} tokens, session average {rate:,.0f} tokens/s')
    print(f'Estimated remaining at this session rate: {(run.target_tokens-tokens)/max(rate,1)/3600:.1f} hours')
    return model, tokens

def check_training_gpu(device, config):
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Training Gu-1B requires a BF16-capable CUDA runtime. Select A100 in Colab.')
    # Test the actual grouped-query FlashAttention kernel and its backward pass before streaming data.
    dim = config.d_model // config.n_heads
    shape = (1, config.n_heads, 16, dim)
    kv_shape = (1, config.n_kv_heads, 16, dim)
    q, k, v = [torch.zeros(s, device=device, dtype=torch.bfloat16, requires_grad=True)
               for s in (shape, kv_shape, kv_shape)]
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)
        output.sum().backward()
    torch.cuda.synchronize(device)

## Start or resume Training Gu-1B

This cell starts the actual training workload. Automatic recovery selects the greatest completed token count with full training state. If only model snapshots exist, it asks you to choose a full checkpoint or explicitly start fresh. Each invocation writes a new run folder.

Training checks BF16 grouped-query FlashAttention forward and backward on the assigned GPU before opening the corpus. This short kernel check does not measure peak memory at long context. To verify a real save on Colab, set `SESSION_STOP_TOKENS=65536` on a fresh run, train one update, and check that the printed checkpoint path appears in Drive. Clear the stop value before continuing.

In [ ]:
config = ModelConfig()
run = RunConfig(target_tokens=TARGET_TOKENS)
resume_path = latest_full(CHECKPOINT_ROOT) if RESUME_FROM == 'auto' else RESUME_FROM
if RESUME_FROM == 'auto' and resume_path is None and checkpoint_files(CHECKPOINT_ROOT):
    raise RuntimeError('Only weights-only checkpoints were found. Use a full checkpoint, or set RESUME_FROM=None to start fresh.')
print('Resume checkpoint', resume_path)
model, tokens_seen = train_gu(config, run, CHECKPOINT_ROOT, resume_from=resume_path,
                               session_stop=SESSION_STOP_TOKENS, session_hours=SESSION_HOURS)

## Validation

Run this after a session ends to measure held-out loss. It uses a separate streaming source and leaves the training cursor unchanged. The default evaluates eight 4k sequences (32,768 held-out targets) using the data revision recorded on the model in memory. Long-context usefulness needs additional tasks with dependencies spanning the target window.

In [ ]:
def evaluate_gu(model, batches=8, sequence_length=4096, data_factory=TokenStream):
    if min(batches, sequence_length) <= 0:
        raise ValueError('Evaluation needs a positive batch count and context length')
    validation = data_factory(model.source_metadata, validation=True)
    device = next(model.parameters()).device
    length = min(sequence_length, model.config.max_seq_len)
    was_training = model.training
    model.eval()
    total_loss, total_tokens = 0.0, 0
    try:
        with torch.no_grad():
            for _ in range(batches):
                x, y = validation.batch(length, device)
                amp = torch.autocast('cuda', dtype=torch.bfloat16) if device.type == 'cuda' else nullcontext()
                with amp:
                    total_loss += model.loss_sum(x, y).item()
                total_tokens += y.numel()
    finally:
        model.train(was_training)
    avg = total_loss / total_tokens
    return {'tokens': total_tokens, 'loss': avg, 'perplexity': math.exp(avg) if avg < 700 else math.inf}

RUN_VALIDATION = False
if RUN_VALIDATION:
    print(evaluate_gu(model))

## Download a model or a resumable checkpoint, or copy it to Drive

Set `EXPORT_ACTION` to `download_model`, `download_resume`, or `copy_to_drive` and run the cell. Select a milestone with `EXPORT_TOKENS`, or keep `None` for the latest applicable save. A resumable ZIP includes its manifest and checkpoint file. Allow enough local disk space for the ZIP. Weights-only files load with `weights_only=True`; full files contain trusted Python/NumPy training state and require `weights_only=False`.

In [ ]:
def export_gu(info, selected, action, output_dir, drive_root=None):
    selected = Path(selected)
    output = Path(output_dir)
    if action == 'copy_to_drive':
        destination = Path(drive_root) / selected.parent.parent.name / selected.parent.name
        if destination.resolve() == selected.parent.resolve():
            return selected
        if destination.exists():
            raise FileExistsError(f'A checkpoint already exists at {destination}')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + '.incomplete-' + uuid.uuid4().hex[:8])
        shutil.copytree(selected.parent, temporary)
        if (temporary / 'checkpoint.pt').stat().st_size != info['bytes']:
            raise IOError('Drive copy has an unexpected file size')
        temporary.rename(destination)
        return destination / 'checkpoint.pt'

    output.mkdir(parents=True, exist_ok=True)
    name = f"Gu-1B-{info['tokens']}"
    if action == 'download_resume':
        if not info['resumable']:
            raise ValueError('The selected checkpoint contains inference weights only')
        exported = output / (name + '-resume.zip')
    elif action == 'download_model':
        exported = output / (name + '-model.pt')
    else:
        raise ValueError('Unknown export action')
    temporary = exported.with_name(exported.name + '.partial-' + uuid.uuid4().hex[:8])
    try:
        if action == 'download_resume':
            with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
                archive.write(selected, arcname='checkpoint.pt')
                archive.write(selected.parent / 'manifest.json', arcname='manifest.json')
        elif not info['resumable']:
            shutil.copyfile(selected, temporary)
        else:
            payload = checkpoint_payload(selected, weights_only=False)
            inference = {key: payload[key] for key in ('name', 'config', 'source', 'tokens', 'versions')}
            inference['resumable'] = False
            inference['weights'] = {key: value.to(torch.bfloat16) for key, value in payload['weights'].items()}
            torch.save(inference, temporary)
        temporary.replace(exported)
    finally:
        temporary.unlink(missing_ok=True)
    return exported

EXPORT_ACTION = 'list'
EXPORT_TOKENS = None  # For example 500_000_000.
EXPORT_RUN = None  # Optional run-folder name from the listing below.

available = checkpoint_files(CHECKPOINT_ROOT)
for info, path in available:
    print(f"{info['tokens']:,} tokens | resumable={info['resumable']} | {path}")
if EXPORT_ACTION != 'list':
    choices = [(info, path) for info, path in available
               if (EXPORT_RUN is None or path.parent.parent.name == EXPORT_RUN)
               and (EXPORT_TOKENS is None or info['tokens'] == EXPORT_TOKENS)
               and (EXPORT_ACTION != 'download_resume' or info['resumable'])]
    if EXPORT_RUN is None and 'model' in globals() and hasattr(model, 'training_id'):
        choices = [(i, p) for i, p in choices if i.get('training_id', 'legacy') == model.training_id]
    if not choices:
        raise FileNotFoundError('No matching committed checkpoint')
    if len({i.get('training_id', 'legacy') for i, _ in choices}) > 1:
        raise ValueError('Set EXPORT_RUN to select one of the independent training runs listed above')
    info, selected = choices[-1]
    if EXPORT_ACTION == 'copy_to_drive':
        from google.colab import drive
        drive.mount('/content/drive')
    exported = export_gu(info, selected, EXPORT_ACTION, '/content', '/content/drive/MyDrive/Gu-1B/checkpoints')
    print('Exported', exported)
    if EXPORT_ACTION.startswith('download_'):
        from google.colab import files
        files.download(str(exported))

## Load a downloaded Gu-1B model and generate text

Run the architecture definition cells first. This optional cell uses the tokenizer commit recorded with the weights and only projects the last hidden position into vocabulary logits. Generation currently recomputes the prefix each step, so long-prompt generation is slower than an implementation with a KV cache.

Use a downloaded `-model.pt` export or a weights-only milestone. Run setup and the model and checkpoint definition cells. CPU inference is supported for checking an export after disconnecting from the GPU.

CUDA inference uses native BF16 where supported and FP16 on GPUs such as T4. Attention permits a math fallback when FlashAttention is unavailable. The fallback uses more memory at long context. Start with short prompts when testing. Training still requires its BF16 FlashAttention kernel check.

In [ ]:
def load_gu_inference(path, device):
    payload = checkpoint_payload(path, weights_only=True)
    with torch.device('meta'):
        inference_model = GuModel(ModelConfig(**payload['config']))
    inference_model.load_state_dict(payload['weights'], assign=True)
    dtype = torch.float32
    if device.type == 'cuda':
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
        dtype = torch.bfloat16 if native_bf16 else torch.float16
    inference_model = inference_model.to(device=device, dtype=dtype).eval()
    tokenizer = AutoTokenizer.from_pretrained(payload['source']['tokenizer_id'],
                                              revision=payload['source']['tokenizer_revision'])
    return inference_model, tokenizer

@torch.inference_mode()
def generate_gu(inference_model, tokenizer, prompt, max_new_tokens=100):
    device = next(inference_model.parameters()).device
    ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    if ids.numel() == 0:
        raise ValueError('Enter a prompt containing at least one token')
    for _ in range(max_new_tokens):
        context = ids[:, -inference_model.config.max_seq_len:]
        hidden = inference_model.hidden(context)[:, -1, :]
        logits = F.linear(hidden, inference_model.embedding.weight).float() / 0.8
        values, indices = logits.topk(min(50, logits.shape[-1]))
        token = indices.gather(-1, torch.multinomial(values.softmax(-1), 1))
        ids = torch.cat((ids, token), dim=1)
        if token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0].tolist(), skip_special_tokens=True)

MODEL_FILE = None  # Uploaded model export, e.g. '/content/Gu-1B-1000000000-model.pt'.
if MODEL_FILE:
    gpu_ok = torch.cuda.is_available()
    device = torch.device('cuda' if gpu_ok else 'cpu')
    inference_model, tokenizer = load_gu_inference(MODEL_FILE, device)
    print(generate_gu(inference_model, tokenizer, 'The future of artificial intelligence'))